# TP 3 : Retrieval-Augmented Generation (RAG)

## Partie 1 : Extraction de contenu d'un fichier PDF

### Chargement et extraction de contenu d'un fichier PDF avec LangChain (PyPDFLoader)

In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Najlaa\AppData\Local\Temp\ipykernel_18444\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader = PyPDFLoader("acmecorp-employee-handbook.pdf")

In [4]:
data = loader.load()
print(data)

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment free from

### Segmentation de texte pour préparation au RAG avec LangChain

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(data)

print(len(all_splits))

3


## Transformation des chunks en vecteurs sémantiques avec un modèle Hugging Face : Génération d'embeddings textuels

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings 

In [12]:
embeddings = HuggingFaceEmbeddings( 
model_name="sentence-transformers/all-MiniLM-L6-v2" 
) 


c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Najlaa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. I

### Création d’une base vectorielle en mémoire pour stockage et  recherche d’embeddings 


In [13]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

### Indexation des documents dans la base vectorielle pour recherche sémantique


In [14]:
ids = vector_store.add_documents(documents=all_splits)

### Recherche sémantique dans une base vectorielle pour retrouver les informations pertinentes


In [15]:
results = vector_store.similarity_search(
"How many days of vacation does an employee get in their first year?"
)
print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

## Agent RAG

In [17]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for relevant information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [18]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
)

In [20]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="How many days of vacation does an employee get in their first year?"
            )
        ]
    }
)

print(response['messages'][-1].content)

According to the handbook, in an employee's first year, they accrue 10 days of paid vacation per year (0.833 days per month).


## Partie 2 : Extraction de contenu d'une base de données SQL (SQLite)

### Connexion à une base de données SQL (SQLite) avec LangChain

In [23]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

### Création d'un tool personnalisé pour interroger une base SQL avec un agent

In [24]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        print(f"Executing SQL query: {query}")
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [25]:
sql_query.invoke("SELECT * FROM Artist LIMIT 10")

Executing SQL query: SELECT * FROM Artist LIMIT 10


"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

### Création d'un agent LLM pour interroger une base SQl

In [26]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

In [27]:
model = ChatOllama(
    model="llama3.2:3b"
)

In [ ]:
system_prompt = """
You are a SQL expert.

Rules:
- Only use sql_query tool
- The sql_query tool takes a SQL query as input and returns the result of the query.
- Only use available columns
- If information does not exist, say so
- Do not guess
- You have to return the results in a human readable format,, do not return raw SQL results or a sql query.

Database schema:

Table Artist:
- ArtistId
- Name
"""

In [29]:
agent = create_agent(
    model=model,
    tools=[sql_query],
    system_prompt=system_prompt
)

### Interrogation d'un agent SQL via langage naturel et récupération des résultats

In [30]:
question = HumanMessage(
    content="Give me the first 5 artists in the database"
)

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Executing SQL query: SELECT ArtistId, Name FROM Artist ORDER BY ArtistId LIMIT 5
The first 5 artists in the database are:

1. AC/DC
2. Accept
3. Aerosmith
4. Alanis Morissette
5. Alice In Chains
